# SENTINEL- X Agent Training


In [ ]:
!pip install -q langchain- groq

In [ ]:
import os, torch, numpy as np, pandas as pd, json as js, asyncio, re

In [ ]:
from kaggle_ secrets import UserSecretsClient
GROQ_ KEY = UserSecretsClient(). get_ secret("GROQ_ API_ KEY")
os. environ["GROQ_ API_ KEY"] = GROQ_ KEY

In [ ]:
from langchain_ groq import ChatGroq
llm = ChatGroq(model="llama-3.1-70b- versatile", temperature=0.05, groq_ api_ key=GROQ_ KEY)

In [ ]:
class Agent:
    def __init__(self, ver): self. ver = ver; self. imp = {"v":0.25,"a":0.15,"ld":0.20,"od":0.20,"t":0.10,"s":0.10}; self. m = {"t":0,"c":0,"i":0,"a":0.0}
    def feats( self, e): return {"v":min(e. get("v",0)/900,1),"a":min(e. get("a",0)/45_000,1),"ld":abs(e. get("ld",0)),"od":abs(e. get("od",0)),"t":0,"s":0.5}
    async def cls( self, e):
        f = self. feats(e); sc = sum(f[k]*self. imp[k] for k in f)
        p = f"Analyze: {js. dumps(e)} Score:{sc:.2f}"
        try:
            r = await asyncio. to_ thread(llm. invoke, p)
            m = re. search(r'\{.+\}', r. content, re. DOTALL)
            rs = js. loads( m. group()) if m else {"class":"suspicious","confidence":sc}
            self. m["t"] += 1
            return {"id":e. get("id","?"), "class":rs["class"], "confidence":rs["confidence"], "score":sc}
        except: return {"id":e. get("id","?"), "class":"unknown","confidence":0}
    def fb( self, p, a):
        if p==a: self. m["c"] += 1
        else: self. m["i"] += 1
        t = self. m["c"] + self. m["i"]
        if t>0: self. m["a"] = self. m["c"] / t
ag = Agent("v1- kaggle")

In [ ]:
np. random. seed(42); d=[]
for i in range(200): an = np. random. rand()>0.8
    if an: d. append({"id":f"e{i:04d}","v":int(np. random. randint(0,100)),"a":int(np. random. randint(0,5_000)),"ld":float(np. random. uniform(-0.5,0.5)),"od":float(np. random. uniform(-0.5,0.5)),"label":"anomaly"})
    else: d. append({"id":f"e{i:04d}","v":int(np. random. randint(400,550)),"a":int(np. random. randint(30_000,40_000)),"ld":float(np. random. uniform(-0.5,0.5)),"od":float(np. random. uniform(-0.5,0.5)),"label":"normal"})
df = pd. DataFrame(d)

In [ ]:
from sklearn. model_ selection import train_ test_ split
tr, te = train_ test_ split(df, test_ size=0.2, random_ state=42, stratify= df. label)
c=0
for i, (_, r) in enumerate(tr. iterrows()):
    e = r. to_ dict(); t = e. pop("label")
    rs = await ag. cls(e)
    if rs["class"]==t: c+=1
    ag. fb(rs["class"], t)
    if (i+1)%25==0: print(f"{i+1}/{len(tr)} acc={c/(i+1):.0%}")
    asyncio. sleep(0.6)
print( f"Train acc: {c/len(tr):.0%}")

In [ ]:
from sklearn. metrics import accuracy_ score
yt, yp = [], []
for _, r in te. iterrows():
    e = r. to_ dict(); t = e. pop("label")
    rs = await ag. cls(e)
    yt. append(t); yp. append(rs["class"])
    asyncio. sleep(0.6)
acc = accuracy_ score(yt, yp)

In [ ]:
import pickle
md = {"version":ag. ver, "importance":ag. imp, "metrics":ag. mx(), "acc":acc}
pk = f"/kaggle/working/clsf_{ag. ver}.pkl"
with open(pk, 'wb') as f: pickle. dump(md, f)
print( f"Saved: {pk}")

In [ ]:
# Push to HF Hub
from kaggle_ secrets import UserSecretsClient
import subprocess
hf = UserSecretsClient(). get_ secret("HF_ TOKEN")
subprocess. run(["huggingface- cli","login","-- token", hf], check= True)
subprocess. run(["huggingface- cli","upload", f"sentinel- x-{ag. ver}", pk, "-- repo- type", "model"], check= True)